In [1]:
import pandas as pd
import re
import numpy as np

/home/i666sapple/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def parse_dataset(file_path):
    sentences = []
    words = []
    pos_tags = []
    ner_tags = []
    malformed_lines = []

    with open(file_path, 'r', encoding='utf-8') as file:
        current_sentence = []
        current_pos = []
        current_ner = []

        for idx, line in enumerate(file):
            line = line.strip()
            if line:
                if '\t' in line:
                    parts = line.split('\t')
                    if len(parts) == 3:
                        token, pos, ner = parts
                        current_sentence.append(token)
                        current_pos.append(pos)
                        current_ner.append(ner)
                    else:
                        malformed_lines.append((idx, line))
                else:
                    if current_sentence:
                        sentences.append(" ".join(current_sentence))
                        words.append(current_sentence)
                        pos_tags.append(current_pos)
                        ner_tags.append(current_ner)
                        current_sentence = []
                        current_pos = []
                        current_ner = []
            else:
                if current_sentence:
                    sentences.append(" ".join(current_sentence))
                    words.append(current_sentence)
                    pos_tags.append(current_pos)
                    ner_tags.append(current_ner)
                    current_sentence = []
                    current_pos = []
                    current_ner = []

        if current_sentence:  # To handle the last sentence in the file
            sentences.append(" ".join(current_sentence))
            words.append(current_sentence)
            pos_tags.append(current_pos)
            ner_tags.append(current_ner)

    # Creating the DataFrame
    df = pd.DataFrame({
        'sentence': sentences,
        'words': words,
        'pos': pos_tags,
        'ner': ner_tags
    })

    # Mapping POS and NER tags to integers
    pos_set = set(tag for sublist in df['pos'] for tag in sublist)
    ner_set = set(tag for sublist in df['ner'] for tag in sublist)

    pos_mapping = {tag: idx for idx, tag in enumerate(sorted(pos_set))}
    ner_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_set))}

    df['postagid'] = df['pos'].apply(lambda tags: [pos_mapping[tag] for tag in tags])
    
    # Ensure `ner` tags are strings before applying mappings
    df['nertagid'] = df['ner'].apply(lambda tags: [ner_mapping[tag] if isinstance(tag, str) else tag for tag in tags])

    # Create new columns from `ner` column
    def split_ner_tags(tags):
        before_dash = []
        after_dash = []
        for tag in tags:
            if isinstance(tag, str) and '-' in tag:
                before, after = tag.split('-', 1)
                before_dash.append(before)
                after_dash.append(after)
            else:
                before_dash.append(tag)
                after_dash.append('')
        return before_dash, after_dash

    # Convert back to string representation for splitting
    df['ner_str'] = df['ner'].apply(lambda tags: [tag for tag in tags])
    df[['ner_tag_general', 'ner_tag_pos']] = df['ner_str'].apply(lambda tags: pd.Series(split_ner_tags(tags)))
    df.drop('ner_str', axis=1, inplace=True)

    # Mapping new column values to integers
    ner_tag_general_set = set(df['ner_tag_general'].explode().unique())
    ner_tag_pos_set = set(df['ner_tag_pos'].explode().unique())

    ner_tag_general_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_tag_general_set))}
    ner_tag_pos_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_tag_pos_set))}

    df['ner_tag_general'] = df['ner_tag_general'].apply(lambda tags: [ner_tag_general_mapping.get(tag, -1) for tag in tags])
    df['ner_tag_pos'] = df['ner_tag_pos'].apply(lambda tags: [ner_tag_pos_mapping.get(tag, -1) for tag in tags])

    return df, malformed_lines

file_path = 'Data/data.tsv'
df, malformed_lines = parse_dataset(file_path)
df.head()

,sentence,words,pos,ner,postagid,nertagid,ner_tag_general,ner_tag_pos
0,শনিবার (২৭ আগস্ট) রাতে পটুয়াখালী সদর থানার ভা...,"[শনিবার, (২৭, আগস্ট), রাতে, পটুয়াখালী, সদর, থ...","[NNP, PUNCT, NNP, NNC, NNP, NNC, NNC, ADJ, NNC...","[B-D&T, B-OTH, B-D&T, B-D&T, B-GPE, I-GPE, I-G...","[6, 11, 6, 5, 6, 5, 5, 0, 5, 11, 6, 6, 3, 5, 0...","[0, 7, 0, 0, 2, 13, 13, 8, 18, 7, 8, 18, 7, 7,...","[0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0]","[0, 7, 0, 0, 2, 2, 2, 8, 8, 7, 8, 8, 7, 7, 7, 7]"
1,বায়ুদূষণ ও স্মার্ট ফোন ছেলেমেয়ে উভয়ের প্রজনন ক...,"[বায়ুদূষণ, ও, স্মার্ট, ফোন, ছেলেমেয়ে, উভয়ের, প...","[NNC, CONJ, NNC, NNC, NNC, PRO, NNC, NNC, NNC,...","[B-OTH, B-OTH, B-OTH, B-OTH, B-PER, B-OTH, B-O...","[5, 2, 5, 5, 5, 10, 5, 5, 5, 14, 13]","[7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7]"
2,ছাত্র রাজনীতির বর্তমান অবস্থার শুরু হয়েছিলো ...,"[ছাত্র, রাজনীতির, বর্তমান, অবস্থার, শুরু, হয়ে...","[NNC, NNC, ADJ, NNC, NNC, VF, NNC, NNP, NNC, PP]","[B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-P...","[5, 5, 0, 5, 5, 13, 5, 6, 5, 9]","[7, 7, 7, 7, 7, 7, 8, 8, 7, 7]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[7, 7, 7, 7, 7, 7, 8, 8, 7, 7]"
3,"শাকিল রাজধানীর ৩০০ ফিট, দিয়াবাড়ি ও পূর্বাচল ...","[শাকিল, রাজধানীর, ৩০০, ফিট,, দিয়াবাড়ি, ও, পূ...","[NNP, NNC, QF, NNC, NNP, CONJ, NNP, NNC, NNC, ...","[B-PER, B-OTH, B-LOC, I-LOC, B-LOC, B-OTH, B-L...","[6, 5, 12, 5, 6, 2, 6, 5, 5, 9, 5, 14, 13, 9, ...","[8, 7, 3, 14, 3, 7, 3, 7, 7, 7, 8, 7, 7, 7, 7, 6]","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[8, 7, 3, 3, 3, 7, 3, 7, 7, 7, 8, 7, 7, 7, 7, 6]"
4,সম্প্রতি ক্লাবের নবীন ব্যবস্থাপনা প্রশিক্ষণার্...,"[সম্প্রতি, ক্লাবের, নবীন, ব্যবস্থাপনা, প্রশিক্...","[ADV, NNC, ADJ, NNC, NNC, CONJ, NNC, NNC, PP, ...","[B-OTH, B-ORG, B-OTH, B-OTH, B-PER, B-OTH, B-P...","[1, 5, 0, 5, 5, 2, 5, 5, 9, 0, 13, 5, 5]","[7, 6, 7, 7, 8, 7, 8, 8, 7, 7, 7, 1, 12]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[7, 6, 7, 7, 8, 7, 8, 8, 7, 7, 7, 1, 1]"


In [31]:
# import datasets
# from transformers import BertTokenizer, DataCollatorForTokenClassification, AutoModelForTokenClassification
# # Load model directly
# from transformers import AutoTokenizer, AutoModelForMaskedLM

# tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")
# model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-multilingual-cased")

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

In [47]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

tokenizer = AutoTokenizer.from_pretrained("sagorsarker/mbert-bengali-ner")
model = AutoModelForTokenClassification.from_pretrained("sagorsarker/mbert-bengali-ner")

In [48]:
words_list = df['words'][0]
token_input = tokenizer(words_list, is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(token_input.input_ids)
word_ids = token_input.word_ids()

In [49]:
len(df['words'][0])

16

In [50]:
len(tokens)

48

In [51]:
# def tokenize_and_align_labels(examples, label_all_tokens=True): 
#     tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True) 
#     labels = [] 
#     for i, label in enumerate(examples["ner_tags"]): 
#         word_ids = tokenized_inputs.word_ids(batch_index=i) 
#         # word_ids() => Return a list mapping the tokens
#         # to their actual word in the initial sentence.
#         # It Returns a list indicating the word corresponding to each token. 
#         previous_word_idx = None 
#         label_ids = []
#         # Special tokens like `` and `<\s>` are originally mapped to None 
#         # We need to set the label to -100 so they are automatically ignored in the loss function.
#         for word_idx in word_ids: 
#             if word_idx is None: 
#                 # set –100 as the label for these special tokens
#                 label_ids.append(-100)
#             # For the other tokens in a word, we set the label to either the current label or -100, depending on
#             # the label_all_tokens flag.
#             elif word_idx != previous_word_idx:
#                 # if current word_idx is != prev then its the most regular case
#                 # and add the corresponding token                 
#                 label_ids.append(label[word_idx]) 
#             else: 
#                 # to take care of sub-words which have the same word_idx
#                 # set -100 as well for them, but only if label_all_tokens == False
#                 label_ids.append(label[word_idx] if label_all_tokens else -100) 
#                 # mask the subword representations after the first subword
                 
#             previous_word_idx = word_idx 
#         labels.append(label_ids) 
#     tokenized_inputs["labels"] = labels 
#     return tokenized_inputs

In [52]:
# q = tokenize_and_align_labels(df['words'][4:5])

In [53]:
print(df.head())  # Displays the first few rows of the DataFrame
print(df.columns)  # Displays the names of the columns in the DataFrame

                                            sentence  \
0  শনিবার (২৭ আগস্ট) রাতে পটুয়াখালী সদর থানার ভা...   
1  বায়ুদূষণ ও স্মার্ট ফোন ছেলেমেয়ে উভয়ের প্রজনন ক...   
2  ছাত্র রাজনীতির বর্তমান অবস্থার শুরু হয়েছিলো ...   
3  শাকিল রাজধানীর ৩০০ ফিট, দিয়াবাড়ি ও পূর্বাচল ...   
4  সম্প্রতি ক্লাবের নবীন ব্যবস্থাপনা প্রশিক্ষণার্...   

                                               words  \
0  [শনিবার, (২৭, আগস্ট), রাতে, পটুয়াখালী, সদর, থ...   
1  [বায়ুদূষণ, ও, স্মার্ট, ফোন, ছেলেমেয়ে, উভয়ের, প...   
2  [ছাত্র, রাজনীতির, বর্তমান, অবস্থার, শুরু, হয়ে...   
3  [শাকিল, রাজধানীর, ৩০০, ফিট,, দিয়াবাড়ি, ও, পূ...   
4  [সম্প্রতি, ক্লাবের, নবীন, ব্যবস্থাপনা, প্রশিক্...   

                                                 pos  \
0  [NNP, PUNCT, NNP, NNC, NNP, NNC, NNC, ADJ, NNC...   
1  [NNC, CONJ, NNC, NNC, NNC, PRO, NNC, NNC, NNC,...   
2   [NNC, NNC, ADJ, NNC, NNC, VF, NNC, NNP, NNC, PP]   
3  [NNP, NNC, QF, NNC, NNP, CONJ, NNP, NNC, NNC, ...   
4  [ADV, NNC, ADJ, NNC, NNC, CONJ, NNC, NNC, P

In [58]:
# Assuming df is your DataFrame
new_df = pd.DataFrame(df)

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

In [84]:
# Tokenize and align labels
def tokenize_and_align_labels(row, label_all_tokens=True):
    tokens = row['words']
    ner_tags = row['nertagid']
    
    tokenized_inputs = tokenizer(tokens, truncation=True, is_split_into_words=True)
    
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []
    
    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(ner_tags[word_idx])
        else:
            label_ids.append(ner_tags[word_idx] if label_all_tokens else -100)
        previous_word_idx = word_idx
    
    tokenized_inputs["labels"] = label_ids
    return pd.Series({
        'input_ids': tokenized_inputs['input_ids'],
        'attention_mask': tokenized_inputs['attention_mask'],
        'labels': label_ids
    })

In [85]:
tokenized_df = new_df.apply(tokenize_and_align_labels, axis=1)

In [86]:
#Define training args
from transformers import TrainingArguments, Trainer 


args = TrainingArguments( 
"test-ner",
evaluation_strategy = "epoch", 
learning_rate=2e-5, 
per_device_train_batch_size=16, 
per_device_eval_batch_size=16, 
num_train_epochs=3, 
weight_decay=0.01, 
)

/home/i666sapple/.local/lib/python3.12/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [103]:
data_collator = DataCollatorForTokenClassification(tokenizer) 

In [88]:
from datasets import Dataset, load_metric

metric = load_metric("seqeval", trust_remote_code=True)

In [89]:
example = new_df.iloc[0]

In [90]:
label_list = sorted(new_df['nertagid'].explode().unique().tolist())


In [91]:
for i in example['nertagid']:
    print(i)

0
7
0
0
2
13
13
8
18
7
8
18
7
7
7
7


In [93]:
labels = [label_list[i] for i in example['nertagid']]
labels

[0, 7, 0, 0, 2, 13, 13, 8, 18, 7, 8, 18, 7, 7, 7, 7]

In [94]:
metric.compute(predictions=[labels], references=[labels])

/home/i666sapple/.local/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/i666sapple/.local/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 7 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/i666sapple/.local/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 2 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/i666sapple/.local/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 13 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/i666sapple/.local/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 8 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/i666sapple/.local/lib/p

{'3': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 '8': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2},
 '_': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 3},
 'overall_precision': 1.0,
 'overall_recall': 1.0,
 'overall_f1': 1.0,
 'overall_accuracy': 1.0}

In [95]:
def compute_metrics(eval_preds): 
    pred_logits, labels = eval_preds 
    
    pred_logits = np.argmax(pred_logits, axis=2) 
    # the logits and the probabilities are in the same order,
    # so we don’t need to apply the softmax
    
    # We remove all the values where the label is -100
    predictions = [ 
        [label_list[eval_preds] for (eval_preds, l) in zip(prediction, label) if l != -100] 
        for prediction, label in zip(pred_logits, labels) 
    ] 
    
    true_labels = [ 
      [label_list[l] for (eval_preds, l) in zip(prediction, label) if l != -100] 
       for prediction, label in zip(pred_logits, labels) 
   ] 
    results = metric.compute(predictions=predictions, references=true_labels)

    return { 
          "precision": results["overall_precision"], 
          "recall": results["overall_recall"], 
          "f1": results["overall_f1"], 
          "accuracy": results["overall_accuracy"], 
    }

In [99]:
tokenized_df.columns

Index(['input_ids', 'attention_mask', 'labels'], dtype='object')

In [100]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [104]:
trainer = Trainer( 
   model, 
   args, 
   train_dataset=train_df, 
   eval_dataset=test_df, 
   data_collator=data_collator, 
   tokenizer=tokenizer, 
   compute_metrics=compute_metrics 
) 

In [105]:
trainer.train()

  0%|          | 0/1053 [00:00<?, ?it/s]

KeyError: 2337

In [107]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, TrainingArguments, Trainer, DataCollatorForTokenClassification, AutoModelForTokenClassification
from datasets import Dataset, load_metric
from sklearn.model_selection import train_test_split

# Assuming df is your DataFrame
new_df = pd.DataFrame(df)

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

# Tokenize and align labels
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["words"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["nertagid"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if True else -100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(new_df)

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# Split the dataset
train_dataset, test_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42).values()

# Define training arguments
args = TrainingArguments(
    "test-ner",
    evaluation_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Create data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# Load the seqeval metric
metric = load_metric("seqeval", trust_remote_code=True)

# Extract unique labels
label_list = sorted(new_df['nertagid'].explode().unique().tolist())
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {v: k for k, v in id2label.items()}

# Define compute_metrics function
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Initialize model
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-multilingual-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

print("Training completed!")

Map:   0%|          | 0/7002 [00:00<?, ? examples/s]

/home/i666sapple/.local/lib/python3.12/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different na

  0%|          | 0/1053 [00:00<?, ?it/s]

KeyboardInterrupt: 